# frugal-eval quickstart

Runs on real per-question results of 395 Open LLM Leaderboard models, bundled with the package. **No API key needed.**

Three problems:
1. **Find the best** prompt or model with as few calls as possible.
2. **Estimate** every model's score from a small subset of questions.
3. **Online**: pick a model for each incoming request while learning.

In [ ]:
!pip install -q git+https://github.com/raja-sunkara/frugal-eval

In [ ]:
import numpy as np
import frugal_eval as fe

for m in fe.list_methods():
    print(f"{m.problem:9} {m.name:24} {m.paper}")

## A real task

10 random models from the leaderboard, evaluated on GSM8K (1,319 math questions). The full results are hidden from the methods; they only see what they ask for.

In [ ]:
task = fe.tasks.make("leaderboard/gsm8k", n_candidates=10, seed=0)
print(task.n_candidates, "models,", task.n_examples, "questions")
print("True best:", task.names[int(np.argmax(task.truth))])

## 1. Find the best model with fewer calls

In [ ]:
result = fe.identify(task, method="lucb", delta=0.05)
print(result.summary())

In [ ]:
for m in fe.list_methods("identify"):
    budget = int(0.3 * task.n_candidates * task.n_examples) if m.kind == "fixed budget" else None
    r = fe.identify(task, method=m.name, budget=budget)
    print(f"{m.name:24} saved {r.savings:4.0%}   picked the true best: {r.best == int(np.argmax(task.truth))}")

## 2. Estimate every score from 100 questions

`irt` and `anchor_points` learn which questions are informative from *other* models' results (`task.prior`).

In [ ]:
for m in fe.list_methods("estimate"):
    r = fe.estimate(task, method=m.name, budget=100)
    print(f"{m.name:16} mean abs error {r.mae*100:4.1f} points")
print()
print(fe.estimate(task, method="irt", budget=100).summary())

## 3. Online selection and routing

Each question arrives once; the method picks one model to answer it and only sees that model's result. Contextual methods (`linucb`, `lin_ts`) route by question features and can beat every single model.

In [ ]:
for m in fe.list_methods("online"):
    r = fe.run_online(task, method=m.name)
    print(f"{m.name:16} average {r.average_reward:.3f}  ({r.total_reward / r.best_fixed_reward:.0%} of the best single model)")

## Your own prompts or models (live)

Write one function that runs a candidate on one example and returns a score in [0, 1]. Works with any OpenAI-compatible API.

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = "..."
# from frugal_eval.llm import chat, last_number
#
# candidates = [
#     {"model": "gpt-4o-mini", "prompt": "Answer the question.\n\n{q}"},
#     {"model": "gpt-4o-mini", "prompt": "Think step by step, then give the final number.\n\n{q}"},
# ]
# test_set = [{"q": "...", "answer": "42"}]
#
# def run_and_grade(candidate, example):
#     reply = chat(candidate["model"], candidate["prompt"].format(q=example["q"]))
#     return last_number(reply, example["answer"])
#
# live = fe.tasks.from_function(candidates, test_set, run_and_grade, cache_path="cache.db")
# print(fe.identify(live, method="lucb").summary())

## Add your own method

```python
@fe.register("identify", "my_method", paper="Me et al. (2026)")
def my_method(ev, budget=None, **opts):
    ...  # call ev.pull([...]) and return ev.result(...)
```
It then runs everywhere, including the benchmark.